# OntodockerClient usage

#### set up a fake session that behaves like an Ontodocker service
Only for demonstration in this notebook.

In [1]:
from tempfile import TemporaryDirectory
from pathlib import Path
from types import SimpleNamespace
import pandas as pd

from courier.transport.url import join_url

try:
    # Optional helper module (may exist in interactive/dev setups).
    from notebooks.fake_http_session import FakeResponse, FakeSession
except ImportError:  # pragma: no cover
    import requests

    class FakeResponse:
        def __init__(self, *, url: str, status_code: int = 200, text: str = "ok"):
            self.url = url
            self.status_code = status_code
            self.text = text
            self.request = None

        def raise_for_status(self):
            if self.status_code >= 400:
                raise requests.HTTPError(f"HTTP {self.status_code}")

    class FakeSession:
        def __init__(self, *, routes: dict[tuple[str, str], FakeResponse]):
            self.routes = routes
            self.headers: dict[str, str] = {}
            self.calls: list[dict] = []

        def request(self, **kwargs):
            method = kwargs.get("method")
            url = kwargs.get("url")
            headers = kwargs.get("headers")

            # mimic requests.Session header merge
            merged = dict(self.headers)
            if headers:
                merged.update(headers)
            kwargs["headers"] = merged

            self.calls.append(kwargs)

            resp = self.routes[(method, url)]
            resp.request = SimpleNamespace(method=method)
            return resp


In [2]:
base_url = "https://example.org"
endpoints_url = join_url(base_url, segments=["api", "v1", "endpoints"])
dataset_url = join_url(base_url, segments=["api", "v1", "jena", "dataset"])
sparql_url = join_url(base_url, segments=["api", "v1", "jena", "dataset", "sparql"])

routes = {
    ("GET", endpoints_url): FakeResponse(
        url=endpoints_url,
        status_code=200,
        # legacy-style endpoint list, intentionally malformed to exercise rectification
        text="['http://example.org:None/api/jena/dataset/sparql']",
    ),
    ("PUT", dataset_url): FakeResponse(url=dataset_url, status_code=200, text='created'),
    ("DELETE", dataset_url): FakeResponse(url=dataset_url, status_code=200, text='deleted'),
    ("GET", dataset_url): FakeResponse(url=dataset_url, status_code=200, text='@prefix : <x> .'),
    ("POST", dataset_url): FakeResponse(url=dataset_url, status_code=200, text='upload ok'),
    ("GET", sparql_url): FakeResponse(
        url=sparql_url,
        status_code=200,
        text='{"results": {"bindings": [{"a": {"value": "1"}, "b": {"value": "2"}}]}}',
    ),
}

fake = FakeSession(routes=routes)

In [3]:
# In older versions, query_df used SPARQLWrapper.
# It now uses query_raw + JSON parsing, so we just provide a SPARQL JSON result
# at the fake SPARQL endpoint route (see below).

## Using `OntodockerClient`

Instantiation

By default, `OntodockerClient` verifies TLS certificates when connecting over HTTPS. This is the
recommended and secure behavior. Only in controlled/local development environments (for example,
when using self-signed certificates) should you consider disabling verification via a `verify=False`
argument, and this must **never** be used in production.

In [4]:
from courier.services.ontodocker import OntodockerClient

token = "<TOKEN>"
address = "example.org"

client = OntodockerClient(address, token=token, timeout=(1.0, 2.0), session=fake)

### List available endpoints
Endpoints are written to a dataclass `EndpointResource`. You can list them "raw" (`list[str]` containing endpoints), or `EndpointInfo` objects containing dataset name and SPARQL endpoint URL.

In [5]:
client.endpoints.list_raw()

['https://example.org/api/v1/jena/dataset/sparql']

In [6]:
client.endpoints.list()

[EndpointInfo(dataset='dataset', sparql_endpoint='https://example.org/api/v1/jena/dataset/sparql')]

### Operations regarding datasets

#### List
Get a `list[str]` of all dataset names.

In [7]:
client.datasets.list()

['dataset']

#### Create a dataset
The specific response message of a real ontodocker looks a bit different.  
**Note:** If a dataset with the specified name already exists, creation fails and an error is forwarded.

In [8]:
client.datasets.create(name="dataset")

'created'

#### Upload a turtlefile

In [9]:
with TemporaryDirectory() as tmp:
    p = Path(tmp) / "in.ttl"
    p.write_text("@prefix : <x> .", encoding="utf-8") # just for the mock-up
    
    resp = client.datasets.upload_turtlefile(name="dataset", turtlefile=str(p))
resp

'upload ok'

#### Download a turtlefile

In [10]:
ttl = client.datasets.download_turtle(name="dataset")

### SPARQL queries

You can make "raw" queries and only return the response as text:

In [11]:
# sparql.query_raw (uses HttpClient under the hood; fully offline here)
client.sparql.query_raw(dataset="dataset", query="SELECT * WHERE { ?s ?p ?o }")

'{}'

Or make responses which are directly written to a pandas dataframe, with an optional list of column headers

In [12]:
# query_df now uses query_raw + JSON parsing under the hood.
df = client.sparql.query_df("dataset", "SELECT ?a ?b WHERE {}", columns=["a", "b"])
df

,a,b
0,1,2
